# Exploring DocMind tokenization and BM25+

This notebook is a small, executable tour of the sparse retrieval layer. It shows:

- how English and Italian text is normalized and tokenized;
- how ingestion `TextChunk` objects are added to the BM25+ index;
- the exact dictionary structure returned by `index.search`;
- metadata filtering, unknown terms, and upserting an existing chunk.

Run this notebook from the repository root so that the `app` package is importable.

In [ ]:
from pathlib import Path
import sys
from pprint import pprint

# Make the notebook work whether Jupyter was launched from the repository
# root or from the notebooks/ directory. Moving the notebook is not needed.
project_root = next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
     if (candidate / 'app').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Launch Jupyter from inside the DocMind repository')
sys.path.insert(0, str(project_root))

from app.ingestion.text import TextChunk
from app.indexing import BM25Index, TokenizerConfig, normalize_text, tokenize

## 1. Normalize and tokenize text

The same tokenizer is used for indexed chunks and user queries. By default it applies Unicode normalization, lowercasing, accent folding, punctuation splitting, and English/Italian stopword removal.

In [ ]:
english_text = "The Café project is running quickly!"
italian_text = "La riunione è nella sala dell'università."

print('English normalized:', normalize_text(english_text))
print('English tokens:    ', tokenize(english_text))
print()
print('Italian normalized:', normalize_text(italian_text))
print('Italian tokens:    ', tokenize(italian_text))

In [ ]:
# Stopword removal can be disabled when function words are meaningful.
without_stopwords = TokenizerConfig(remove_stopwords=False)
print(tokenize("The project e la riunione", without_stopwords))

# Or restrict the built-in stopword language selection.
italian_only = TokenizerConfig(languages=("it",))
print(tokenize("The project e la riunione", italian_only))

## 2. Create representative ingestion chunks

In the application these objects come from `ingest_file(...)`. Here they are created directly so the experiment stays small and reproducible. Metadata is kept for citations and later filtering.

In [ ]:
chunks = [
    TextChunk(
        text="The project deadline is Friday. The team will review the release plan.",
        source="meeting-notes.md",
        chunk_id="meeting-notes:chunk-0",
        metadata={"file_type": "md", "page": "1"},
    ),
    TextChunk(
        text="La riunione del progetto è prevista per venerdì nella sala principale.",
        source="verbale-riunione.md",
        chunk_id="verbale-riunione:chunk-0",
        metadata={"file_type": "md", "page": "2"},
    ),
    TextChunk(
        text="The recipe uses flour, water, and olive oil.",
        source="cooking-notes.txt",
        chunk_id="cooking-notes:chunk-0",
        metadata={"file_type": "txt"},
    ),
]

for chunk in chunks:
    print(chunk.chunk_id, '->', chunk.text)

## 3. Build and inspect the BM25+ index

In [ ]:
index = BM25Index()
index.add_chunks(chunks)
print('Number of indexed chunks:', index.size)
print('Stored chunk IDs:', [chunk.chunk_id for chunk in index.chunks])

## 4. Search and inspect the result structure

`index.search(...)` returns a list of dictionaries. Each dictionary contains:

- `id` and `chunk_id`: stable chunk identifiers;
- `text`: original chunk text;
- `source`: source path;
- `metadata`: page, file type, timestamps, or other ingestion metadata;
- `score`: BM25+ relevance score.

The exact output can be inspected in the following cells.

In [ ]:
results = index.search("project deadline", top_k=3)
pprint(results)

In [ ]:
# Inspect the Python types and keys of the first result.
first_result = results[0]
print('Result container type:', type(results))
print('Single result type:   ', type(first_result))
print('Result keys:           ', list(first_result))
print('Metadata type:         ', type(first_result['metadata']))
print('Score type:            ', type(first_result['score']))

In [ ]:
# A compact view is often easier to read than the complete dictionaries.
for rank, result in enumerate(results, start=1):
    print(f"{rank}. score={result['score']:.4f} | {result['source']} | {result['text']}")

## 5. Metadata filtering

Filtering does not alter the BM25 corpus statistics. All documents are scored first, and the metadata filter is applied to the ranked results.

In [ ]:
italian_results = index.search("progetto riunione", metadata_filter={"page": "2"})
pprint(italian_results)

## 6. Edge cases and upserts

In [ ]:
print('Unknown query:', index.search('quantum spaceship'))

# Re-adding an existing chunk_id replaces the old content instead of
# duplicating it and distorting term frequencies.
index.add_chunks([TextChunk(
    text="The project deadline moved to Monday.",
    source="meeting-notes.md",
    chunk_id="meeting-notes:chunk-0",
    metadata={"file_type": "md", "page": "1"},
)])
print('Size after upsert:', index.size)
pprint(index.search('deadline Monday', top_k=1))